In [1]:
from bs4 import BeautifulSoup
from tqdm import tqdm

import itertools
import json
import pandas as pd 
import pickle
import re
import requests 

In [2]:
url = 'https://locations.traderjoes.com/'
page = requests.get(url)

In [3]:
page

<Response [200]>

In [4]:
soup = BeautifulSoup(page.content, 'html.parser')

In [5]:
# Section of the website which contains list of states w/ TJs
results = soup.find(id='contentbegin')

In [6]:
states = [link.get('href') for link in tqdm(results.find_all('a'))]

100%|██████████| 43/43 [00:00<00:00, 171277.37it/s]


In [6]:
def get_content(url):
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    results = soup.find(id='contentbegin')
    links = [link.get('href') for link in results.find_all('a')]
    return links

In [23]:
states = get_content(url)

In [27]:
cities = [get_content(state) for state in states]

In [28]:
flat_cities = list(itertools.chain(*cities))

In [29]:
store_links = [get_content(city) for city in tqdm(flat_cities)]

100%|██████████| 415/415 [06:17<00:00,  1.10it/s]


In [58]:
all_stores = list(set(list(itertools.chain(*store_links))))

In [18]:
len(all_stores)

524

In [14]:
all_stores[0]

'https://locations.traderjoes.com/nj/denville/617/'

In [65]:
with open('store_links.pkl', 'wb') as f:
    pickle.dump(all_stores, f)

In [3]:
with open('store_links.pkl', 'rb') as f:
    all_stores = pickle.load(f)

In [88]:
all_stores[0]

'https://locations.traderjoes.com/ca/rancho-palos-verdes/233/'

In [89]:
def get_main(url):
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    results = soup.find(id='contentbegin')
    return results

In [167]:
tester = get_main(all_stores[0])

In [170]:
trial = tester.find('div', class_="addressline").get_text().replace('\t', "").strip()
trial

'31176 Hawthorne Blvd\n         \n         Rancho Palos Verdes\n         , \n         CA\n         90275\n         \n\n\n                     310-544-1727\n                  \n\n\n\n                     310-544-1727'

In [171]:
again = trial.split('\n')
again = [a.strip() for a in again]
test_list = list(filter(None, again))
test_list = [x for x in test_list if "," not in x]
test_list

['31176 Hawthorne Blvd',
 'Rancho Palos Verdes',
 'CA',
 '90275',
 '310-544-1727',
 '310-544-1727']

In [140]:
store_name = tester.find('div', class_="h1title").get_text() 
test_list = [store_name] + test_list

In [31]:
def get_name(url):
    '''Get the main section of the page which contains the address information'''
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    results = soup.find(id='contentbegin')
    store_name = results.find('div', class_="h1title").get_text() 

    return store_name

get_name("https://locations.traderjoes.com/al/birmingham/737/")

"Trader Joe's Birmingham (737)"

In [78]:
def get_address(url): 
    """Returns the address information for the specified store

    Args:
        url (str): URL for the store in question

    Returns:
        address (list): Contains the city, state, zip code, landline, and cell phone
    """
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    results = soup.find(id='contentbegin')
    address = results.find('div', class_="addressline").get_text().replace('\t', "").strip().split('\n')
    address = [a.strip() for a in address]
    address = list(filter(None, address))
    # address = [a for a in address if "," not in a]
    # store_name = raw.find('div', class_="h1title").get_text() 
    return  address

In [83]:
test = get_address("https://locations.traderjoes.com/al/birmingham/737/")
test

['205 Summit Blvd, Suite 100',
 'Birmingham',
 ',',
 'AL',
 '35243',
 '205-969-7801',
 '205-969-7801']

In [85]:
test

['205 Summit Blvd, Suite 100',
 'Birmingham',
 'AL',
 '35243',
 '205-969-7801',
 '205-969-7801']

In [88]:
def get_info(url): 
    """
    Returns the information of interest for a given Trader Joe's store

    Args:
        url (list): URL of the store

    Returns:
        store_info  (list): Contains the store's name, city, state, zip, landline, 
                            cell phone, and URL
    """
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    results = soup.find(id='contentbegin')
    store_name = results.find('div', class_="h1title").get_text() 
    address = results.find('div', class_="addressline").get_text().replace('\t', "").strip().split('\n')
    address = [a.strip() for a in address]
    address = list(filter(None, address))
    del address[2] #deletes the comma between city and state
 
    store_info = [store_name] + address +[url]

    return store_info

In [89]:
get_info("https://locations.traderjoes.com/al/birmingham/737/")

["Trader Joe's Birmingham (737)",
 '205 Summit Blvd, Suite 100',
 'Birmingham',
 'AL',
 '35243',
 '205-969-7801',
 '205-969-7801',
 'https://locations.traderjoes.com/al/birmingham/737/']

In [90]:
len(all_stores)

524

In [108]:
all_store_info = [get_info(store) for store in tqdm(all_stores)] 
info = ['store_name','street', 'city', 'state', 'zip', 'landline', 'mobile', 'website']

df = pd.DataFrame(all_store_info, columns=info)    

100%|██████████| 524/524 [09:03<00:00,  1.04s/it]


In [110]:
df.count()

store_name    524
street        524
city          524
state         524
zip           524
landline      524
mobile        521
website       521
dtype: int64

In [111]:
df[df.website.isna()]

,store_name,street,city,state,zip,landline,mobile,website
352,Trader Joe's South Bend (600),1140 East Howard,South Bend,IN,46617,https://locations.traderjoes.com/in/south-bend...,None,None
405,Trader Joe's Crestview Hills Grocery (788),2780 Dixie Highway,Crestview Hills,KY,41017,https://locations.traderjoes.com/ky/crestview-...,None,None
420,Trader Joe's Crestview Hills Wine (789),2785 Dixie Highway,Crestview Hills,KY,41017,https://locations.traderjoes.com/ky/crestview-...,None,None


In [112]:
df.loc[df.website.isnull(), 'website'] = df['landline']

In [114]:
# df.loc[df.mobile.isnull(), 'landline'] = None

In [119]:
df.loc[df.mobile.isnull(), "landline"] = None

In [121]:
df.loc[df.mobile.isnull(), ]

,store_name,street,city,state,zip,landline,mobile,website
352,Trader Joe's South Bend (600),1140 East Howard,South Bend,IN,46617,None,None,https://locations.traderjoes.com/in/south-bend...
405,Trader Joe's Crestview Hills Grocery (788),2780 Dixie Highway,Crestview Hills,KY,41017,None,None,https://locations.traderjoes.com/ky/crestview-...
420,Trader Joe's Crestview Hills Wine (789),2785 Dixie Highway,Crestview Hills,KY,41017,None,None,https://locations.traderjoes.com/ky/crestview-...


In [124]:
df.count()

store_name    524
street        524
city          524
state         524
zip           524
landline      521
mobile        521
website       524
dtype: int64

In [125]:
df.to_csv('store_info.csv', index=False)

In [126]:
df = pd.read_csv('store_info.csv')
df.head()

,store_name,street,city,state,zip,landline,mobile,website
0,Trader Joe's Sunnyvale (68),316 West El Camino Real,Sunnyvale,CA,94087,408-736-7573,408-736-7573,https://locations.traderjoes.com/ca/sunnyvale/68/
1,Trader Joe's Aliso Viejo (195),26541 Aliso Creek Rd,Aliso Viejo,CA,92656,949-643-5531,949-643-5531,https://locations.traderjoes.com/ca/aliso-viej...
2,Trader Joe's Brookline (501),1317 Beacon St,Brookline,MA,02446,617-278-9997,617-278-9997,https://locations.traderjoes.com/ma/brookline/...
3,Trader Joe's Park Ridge (698),190 N NW Highway,Park Ridge,IL,60068,847-292-1108,847-292-1108,https://locations.traderjoes.com/il/park-ridge...
4,Trader Joe's Scottsdale (94),6202 N Scottsdale Rd,Scottsdale,AZ,85253,480-948-9886,480-948-9886,https://locations.traderjoes.com/az/scottsdale...


In [127]:
df.count()

store_name    524
street        524
city          524
state         524
zip           524
landline      521
mobile        521
website       524
dtype: int64